# Semana 2: Tratamiento de Datos Faltantes, Imputación y Detección de Outliers

## Proyecto: Serie de Tiempo de Caudal – Estación Pueblo Nuevo (IDEAM)

**Objetivo:** Convertir la serie con gaps en una serie temporal diaria completa y limpia, lista para EDA avanzado y modelado.

### Contenido:
1. Importar librerías
2. Cargar y preparar datos (replicar pipeline Week 1)
3. Reindexar a frecuencia diaria completa
4. Visualizar el patrón de datos faltantes
5. Métodos de imputación (ffill, interpolación lineal, temporal, media móvil)
6. Comparación visual de métodos y selección
7. Detección de outliers (IQR y Z-score)
8. Tratamiento de outliers (capping / winsorización)
9. Transformaciones (log, Box-Cox, diferenciación)
10. Verificación final de la serie limpia
11. Exportar dataset limpio para Semana 3
12. Conclusiones

## 1. Importar Librerías

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as sp_stats
import warnings
warnings.filterwarnings("ignore")

# Plantilla visual para Plotly
import plotly.io as pio
pio.templates.default = "plotly_white"

print("✅ Librerías importadas")

✅ Librerías importadas


## 2. Cargar y Preparar Datos (Pipeline de la Semana 1)

Replicamos los pasos de limpieza de la Semana 1: cargar CSV → conservar solo `Fecha` y `Valor` → renombrar → establecer DatetimeIndex.

In [2]:
# Cargar datos crudos desde Week_1
df_raw = pd.read_csv("../Week_1/descargaDhime.csv")

# Pipeline de limpieza (Semana 1)
df = (
    df_raw[["Fecha", "Valor"]]
    .copy()
    .rename(columns={"Valor": "Caudal"})
)
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.set_index("Fecha").sort_index()

print(f"✅ Datos cargados desde Week_1")
print(f"   Período: {df.index.min().date()} → {df.index.max().date()}")
print(f"   Registros con dato: {len(df)}")
print(f"   NaN en Caudal: {df['Caudal'].isna().sum()}")
df.head()

✅ Datos cargados desde Week_1
   Período: 2010-01-01 → 2017-12-31
   Registros con dato: 2653
   NaN en Caudal: 0


,Caudal
Fecha,
2010-01-01,1.196
2010-01-02,1.196
2010-01-03,1.196
2010-01-04,1.196
2010-01-05,1.196


## 3. Reindexar a Frecuencia Diaria Completa

La serie actual solo tiene los **días con medición**. Para análisis de series temporales necesitamos un índice diario **continuo** — los días sin registro aparecerán como `NaN`.

In [19]:
# Reindexar a frecuencia diaria completa
rango_completo = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
df_full = df.reindex(rango_completo)
df_full.index.name = "Fecha"

total_dias = len(df_full)
con_dato = df_full["Caudal"].notna().sum()
sin_dato = df_full["Caudal"].isna().sum()

print(f"📅 Serie reindexada a frecuencia diaria:")
print(f"   Total días: {total_dias}")
print(f"   Con dato:   {con_dato} ({con_dato/total_dias*100:.1f}%)")
print(f"   Sin dato (NaN): {sin_dato} ({sin_dato/total_dias*100:.1f}%)")
print(f"   Frecuencia: {df_full.index.freq}")
df_full.head(15)

📅 Serie reindexada a frecuencia diaria:
   Total días: 2922
   Con dato:   2653 (90.8%)
   Sin dato (NaN): 269 (9.2%)
   Frecuencia: <Day>


,Caudal
Fecha,
2010-01-01,1.196
2010-01-02,1.196
2010-01-03,1.196
2010-01-04,1.196
2010-01-05,1.196
2010-01-06,1.196
2010-01-07,1.196
2010-01-08,1.196
2010-01-09,1.196


## 4. Visualizar el Patrón de Datos Faltantes

Antes de imputar, es crucial entender **dónde** y **cuándo** faltan datos. Visualizamos:
- **Mapa de presencia/ausencia** (heatmap binario Año × Mes)
- **Serie con gaps marcados** para ver la distribución temporal de los faltantes

In [4]:
# Heatmap de completitud: % de datos presentes por Año × Mes
df_full_copy = df_full.copy()
df_full_copy["Año"] = df_full_copy.index.year
df_full_copy["Mes"] = df_full_copy.index.month

completitud = df_full_copy.groupby(["Año", "Mes"])["Caudal"].apply(
    lambda x: x.notna().mean() * 100
).reset_index()
completitud.columns = ["Año", "Mes", "Completitud_%"]

meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
pivot_comp = completitud.pivot(index="Año", columns="Mes", values="Completitud_%")
pivot_comp.columns = meses

fig = px.imshow(
    pivot_comp,
    title="Completitud de Datos por Año × Mes (% días con registro)",
    labels=dict(x="Mes", y="Año", color="Completitud (%)"),
    color_continuous_scale="RdYlGn",
    aspect="auto",
    text_auto=".0f",
    zmin=0,
    zmax=100,
)
fig.update_layout(width=900, height=400)
fig.show()

In [5]:
# Serie temporal con gaps resaltados
df_full_copy["Es_NaN"] = df_full_copy["Caudal"].isna().astype(int)

fig = go.Figure()

# Serie con datos
fig.add_trace(go.Scatter(
    x=df_full.index,
    y=df_full["Caudal"],
    mode="lines",
    name="Caudal observado",
    line=dict(color="#1f77b4", width=0.8),
))

# Marcar gaps como barras rojas en el fondo
nan_mask = df_full["Caudal"].isna()
if nan_mask.any():
    y_max = df_full["Caudal"].max() * 1.1
    fig.add_trace(go.Bar(
        x=df_full.index[nan_mask],
        y=[y_max] * nan_mask.sum(),
        name="Días faltantes",
        marker_color="rgba(255, 0, 0, 0.3)",
        width=86400000,  # 1 día en ms
    ))

fig.update_layout(
    title="Serie de Caudal con Gaps Resaltados (rojo = sin dato)",
    xaxis_title="",
    yaxis_title="Caudal (m³/s)",
    width=1000,
    height=500,
    barmode="overlay",
    xaxis=dict(rangeslider=dict(visible=True)),
    hovermode="x unified",
)
fig.show()

In [6]:
# Resumen de gaps consecutivos
fechas_faltantes = df_full.index[df_full["Caudal"].isna()]
gaps_df = pd.DataFrame({"Fecha": fechas_faltantes}).sort_values("Fecha").reset_index(drop=True)
gaps_df["Grupo"] = (gaps_df["Fecha"].diff() > pd.Timedelta(days=1)).cumsum()

gaps_resumen = gaps_df.groupby("Grupo").agg(
    Inicio=("Fecha", "min"),
    Fin=("Fecha", "max"),
    Días=("Fecha", "count"),
).sort_values("Días", ascending=False).reset_index(drop=True)

print(f"📊 Total de gaps consecutivos: {len(gaps_resumen)}")
print(f"📊 Gap más largo: {gaps_resumen['Días'].iloc[0]} días")
print(f"📊 Gap promedio: {gaps_resumen['Días'].mean():.1f} días")
print(f"📊 Mediana de gap: {gaps_resumen['Días'].median():.0f} días\n")

# Distribución de tamaños de gap
fig = px.histogram(
    gaps_resumen,
    x="Días",
    nbins=20,
    title="Distribución del Tamaño de los Gaps (días consecutivos faltantes)",
    labels={"Días": "Tamaño del gap (días)", "count": "Frecuencia"},
    color_discrete_sequence=["#d62728"],
)
fig.update_layout(width=800, height=400)
fig.show()

print("\n🔝 Top 10 gaps más largos:")
gaps_resumen.head(10)

📊 Total de gaps consecutivos: 18
📊 Gap más largo: 111 días
📊 Gap promedio: 14.9 días
📊 Mediana de gap: 2 días




🔝 Top 10 gaps más largos:


,Inicio,Fin,Días
0,2014-11-15,2015-03-05,111
1,2012-11-01,2012-12-31,61
2,2016-08-14,2016-09-14,32
3,2010-01-12,2010-02-12,32
4,2013-12-25,2013-12-31,7
5,2014-08-01,2014-08-05,5
6,2014-08-08,2014-08-11,4
7,2014-07-24,2014-07-26,3
8,2015-06-26,2015-06-28,3
9,2014-02-28,2014-03-01,2


## 5. Métodos de Imputación

Aplicamos **4 métodos** sobre la serie con NaN y los comparamos:

| Método | Descripción | Ventaja | Limitación |
|--------|------------|---------|------------|
| **Forward Fill** (`ffill`) | Propaga el último valor conocido hacia adelante | Simple, preserva nivel | No captura variabilidad, peligroso en gaps largos |
| **Interpolación Lineal** | Traza línea recta entre puntos conocidos | Suave, buen compromiso | Asume cambio uniforme |
| **Interpolación Temporal** (`method='time'`) | Pondera por distancia temporal real | Ideal para fechas irregulares | Similar a lineal si freq=D |
| **Media Móvil** (rolling) | Usa promedio de ventana centrada | Suaviza ruido | Puede perder extremos, no cubre bordes |

In [7]:
# Aplicar los 4 métodos de imputación
df_imp = pd.DataFrame(index=df_full.index)
df_imp["Original"] = df_full["Caudal"]

# 1. Forward Fill (propagar último valor conocido)
df_imp["FFill"] = df_full["Caudal"].ffill()

# 2. Interpolación Lineal
df_imp["Interp_Lineal"] = df_full["Caudal"].interpolate(method="linear")

# 3. Interpolación Temporal
df_imp["Interp_Temporal"] = df_full["Caudal"].interpolate(method="time")

# 4. Media Móvil (rolling de 7 días, luego interpolar residuos)
rolling_7 = df_full["Caudal"].rolling(window=7, center=True, min_periods=1).mean()
df_imp["Media_Movil"] = rolling_7.interpolate(method="linear")

# Verificar NaN residuales en cada método
print("📊 NaN residuales por método:")
print("=" * 40)
for col in df_imp.columns:
    nan_count = df_imp[col].isna().sum()
    print(f"  {col:20s}: {nan_count} NaN")

# Rellenar posibles NaN en bordes con bfill
for col in ["FFill", "Interp_Lineal", "Interp_Temporal", "Media_Movil"]:
    df_imp[col] = df_imp[col].bfill().ffill()

📊 NaN residuales por método:
  Original            : 269 NaN
  FFill               : 0 NaN
  Interp_Lineal       : 0 NaN
  Interp_Temporal     : 0 NaN
  Media_Movil         : 0 NaN


## 6. Comparación Visual de Métodos de Imputación

Visualizamos cómo cada método rellena los gaps, enfocándonos en un **gap representativo** para ver las diferencias.

In [8]:
# Zoom en el gap más largo para comparar métodos
gap_mayor = gaps_resumen.iloc[0]
margen = pd.Timedelta(days=15)
inicio_zoom = gap_mayor["Inicio"] - margen
fin_zoom = gap_mayor["Fin"] + margen

zoom = df_imp.loc[inicio_zoom:fin_zoom]

fig = go.Figure()
colores = {"Original": "#1f77b4", "FFill": "#ff7f0e", "Interp_Lineal": "#2ca02c",
           "Interp_Temporal": "#d62728", "Media_Movil": "#9467bd"}

for metodo, color in colores.items():
    dash = "solid" if metodo == "Original" else "dash"
    width = 2.5 if metodo == "Original" else 1.5
    fig.add_trace(go.Scatter(
        x=zoom.index, y=zoom[metodo],
        mode="lines", name=metodo,
        line=dict(color=color, width=width, dash=dash),
    ))

# Sombrear zona del gap
fig.add_vrect(
    x0=gap_mayor["Inicio"], x1=gap_mayor["Fin"],
    fillcolor="rgba(255,0,0,0.08)", line_width=0,
    annotation_text=f"Gap: {gap_mayor['Días']} días",
    annotation_position="top left",
)

fig.update_layout(
    title=f"Comparación de Métodos de Imputación – Gap más largo ({gap_mayor['Inicio'].date()} a {gap_mayor['Fin'].date()})",
    xaxis_title="", yaxis_title="Caudal (m³/s)",
    width=1000, height=500,
    hovermode="x unified",
)
fig.show()

In [9]:
# Comparación estadística de métodos sobre toda la serie
print("📊 Estadísticas de la serie completa por método de imputación:")
print("=" * 75)
stats_comp = df_imp[["FFill", "Interp_Lineal", "Interp_Temporal", "Media_Movil"]].describe()
display(stats_comp.round(2))

# Diferencia respecto a la media/std del original (solo datos observados)
orig_mean = df_imp["Original"].mean()
orig_std = df_imp["Original"].std()
print(f"\n📌 Original (solo observados): media={orig_mean:.2f}, std={orig_std:.2f}")
print("\n📌 Desviación relativa de cada método vs original:")
for col in ["FFill", "Interp_Lineal", "Interp_Temporal", "Media_Movil"]:
    m = df_imp[col].mean()
    s = df_imp[col].std()
    print(f"  {col:20s}: Δmedia={abs(m-orig_mean):.3f} m³/s, Δstd={abs(s-orig_std):.3f} m³/s")

📊 Estadísticas de la serie completa por método de imputación:


,FFill,Interp_Lineal,Interp_Temporal,Media_Movil
count,2922.00,2922.00,2922.00,2922.00
mean,3.63,3.49,3.49,3.48
std,2.09,1.75,1.75,1.63
min,0.08,0.08,0.08,0.12
25%,2.65,2.63,2.63,2.69
50%,3.33,3.34,3.34,3.37
75%,4.10,4.06,4.06,4.11
max,16.15,16.15,16.15,12.48



📌 Original (solo observados): media=3.40, std=1.61

📌 Desviación relativa de cada método vs original:
  FFill               : Δmedia=0.227 m³/s, Δstd=0.474 m³/s
  Interp_Lineal       : Δmedia=0.082 m³/s, Δstd=0.139 m³/s
  Interp_Temporal     : Δmedia=0.082 m³/s, Δstd=0.139 m³/s
  Media_Movil         : Δmedia=0.077 m³/s, Δstd=0.012 m³/s


### Selección del Método: Interpolación Lineal

**Justificación:**
- Preserva la continuidad de la serie sin crear escalones (como `ffill`)
- Genera transiciones suaves entre puntos conocidos
- Estadísticas (media, varianza) se mantienen cercanas al original
- Para series hidrológicas diarias con gaps moderados, la interpolación lineal es una práctica estándar recomendada por la WMO

Aplicamos **interpolación lineal** como método principal.

In [10]:
# Aplicar interpolación lineal como método seleccionado
df_clean = df_full.copy()
df_clean["Caudal"] = df_clean["Caudal"].interpolate(method="linear").bfill().ffill()

print(f"✅ Imputación aplicada (interpolación lineal)")
print(f"   NaN restantes: {df_clean['Caudal'].isna().sum()}")
print(f"   Total registros: {len(df_clean)}")

# Visualización: antes vs después
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Antes (con gaps)", "Después (interpolación lineal)"],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df_full.index, y=df_full["Caudal"],
              mode="lines", name="Original", line=dict(color="#1f77b4", width=0.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["Caudal"],
              mode="lines", name="Imputado", line=dict(color="#2ca02c", width=0.8)), row=2, col=1)

fig.update_layout(title="Comparación: Serie Original vs Imputada",
                  width=1000, height=600, showlegend=True)
fig.update_yaxes(title_text="Caudal (m³/s)", row=1, col=1)
fig.update_yaxes(title_text="Caudal (m³/s)", row=2, col=1)
fig.show()

✅ Imputación aplicada (interpolación lineal)
   NaN restantes: 0
   Total registros: 2922


## 7. Detección de Outliers

Usamos dos métodos complementarios para identificar valores atípicos:

1. **Método IQR (Rango Intercuartílico):** outlier si $x < Q_1 - 1.5 \times IQR$ o $x > Q_3 + 1.5 \times IQR$
2. **Z-score:** outlier si $|z| > 3$ (más de 3 desviaciones estándar de la media)

In [11]:
# Detección de outliers - Método IQR
Q1 = df_clean["Caudal"].quantile(0.25)
Q3 = df_clean["Caudal"].quantile(0.75)
IQR = Q3 - Q1
lim_inf_iqr = Q1 - 1.5 * IQR
lim_sup_iqr = Q3 + 1.5 * IQR

outliers_iqr = df_clean[(df_clean["Caudal"] < lim_inf_iqr) | (df_clean["Caudal"] > lim_sup_iqr)]

print("📊 Detección de Outliers - Método IQR")
print("=" * 50)
print(f"   Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
print(f"   Límite inferior: {lim_inf_iqr:.2f} m³/s")
print(f"   Límite superior: {lim_sup_iqr:.2f} m³/s")
print(f"   Outliers detectados: {len(outliers_iqr)} ({len(outliers_iqr)/len(df_clean)*100:.1f}%)")

# Detección de outliers - Z-score
df_clean["Zscore"] = np.abs(sp_stats.zscore(df_clean["Caudal"], nan_policy="omit"))
outliers_z = df_clean[df_clean["Zscore"] > 3]

print(f"\n📊 Detección de Outliers - Z-score (|z| > 3)")
print("=" * 50)
print(f"   Outliers detectados: {len(outliers_z)} ({len(outliers_z)/len(df_clean)*100:.1f}%)")

# Resumen combinado
ambos = outliers_iqr.index.intersection(outliers_z.index)
print(f"\n📌 Outliers detectados por AMBOS métodos: {len(ambos)}")

📊 Detección de Outliers - Método IQR
   Q1 = 2.63, Q3 = 4.06, IQR = 1.43
   Límite inferior: 0.49 m³/s
   Límite superior: 6.20 m³/s
   Outliers detectados: 288 (9.9%)

📊 Detección de Outliers - Z-score (|z| > 3)
   Outliers detectados: 53 (1.8%)

📌 Outliers detectados por AMBOS métodos: 53


In [12]:
# Visualización de outliers sobre la serie
fig = go.Figure()

# Serie completa
fig.add_trace(go.Scatter(
    x=df_clean.index, y=df_clean["Caudal"],
    mode="lines", name="Caudal",
    line=dict(color="#1f77b4", width=0.8),
))

# Outliers IQR
fig.add_trace(go.Scatter(
    x=outliers_iqr.index, y=outliers_iqr["Caudal"],
    mode="markers", name=f"Outliers IQR ({len(outliers_iqr)})",
    marker=dict(color="red", size=5, symbol="circle"),
))

# Líneas de límite IQR
fig.add_hline(y=lim_sup_iqr, line_dash="dash", line_color="red",
              annotation_text=f"Límite superior IQR = {lim_sup_iqr:.1f}")
fig.add_hline(y=lim_inf_iqr, line_dash="dash", line_color="orange",
              annotation_text=f"Límite inferior IQR = {lim_inf_iqr:.1f}")

fig.update_layout(
    title="Serie de Caudal con Outliers Detectados (Método IQR)",
    xaxis_title="", yaxis_title="Caudal (m³/s)",
    width=1000, height=500,
    hovermode="x unified",
)
fig.show()

# Boxplot con outliers marcados
fig_box = px.box(
    df_clean.reset_index(), y="Caudal",
    title="Boxplot del Caudal (post-imputación)",
    points="outliers",
    color_discrete_sequence=["#ff7f0e"],
)
fig_box.update_layout(width=400, height=450)
fig_box.show()

## 8. Tratamiento de Outliers: Capping (Winsorización)

En series hidrológicas, los **valores extremos altos** suelen ser **crecientes reales** (eventos legítimos).  
En lugar de eliminarlos, aplicamos **capping** (winsorización): recortamos los valores que exceden los percentiles 1 y 99. Esto conserva la forma de la serie pero limita la influencia de extremos.

> **Nota:** Conservamos también la serie sin tratar (`Caudal_sin_cap`) para comparación.

In [13]:
# Capping / Winsorización en percentiles 1-99
p01 = df_clean["Caudal"].quantile(0.01)
p99 = df_clean["Caudal"].quantile(0.99)

df_clean["Caudal_sin_cap"] = df_clean["Caudal"].copy()
df_clean["Caudal"] = df_clean["Caudal"].clip(lower=p01, upper=p99)

capped = (df_clean["Caudal_sin_cap"] != df_clean["Caudal"]).sum()
print(f"✅ Capping aplicado (percentiles 1-99)")
print(f"   Límite inferior (P1):  {p01:.2f} m³/s")
print(f"   Límite superior (P99): {p99:.2f} m³/s")
print(f"   Valores recortados:    {capped}")

# Comparación antes/después del capping
fig = make_subplots(rows=1, cols=2, subplot_titles=["Antes del capping", "Después del capping"])

fig.add_trace(go.Histogram(x=df_clean["Caudal_sin_cap"], nbinsx=50,
              marker_color="#ff7f0e", name="Sin capping"), row=1, col=1)
fig.add_trace(go.Histogram(x=df_clean["Caudal"], nbinsx=50,
              marker_color="#2ca02c", name="Con capping"), row=1, col=2)

fig.update_layout(title="Distribución del Caudal: Antes vs Después del Capping",
                  width=900, height=400, showlegend=True)
fig.update_xaxes(title_text="Caudal (m³/s)")
fig.update_yaxes(title_text="Frecuencia")
fig.show()

✅ Capping aplicado (percentiles 1-99)
   Límite inferior (P1):  0.15 m³/s
   Límite superior (P99): 9.39 m³/s
   Valores recortados:    59


## 9. Transformaciones: Log, Box-Cox y Diferenciación

Las transformaciones ayudan a:
- **Estabilizar la varianza** (log, Box-Cox)
- **Lograr normalidad aproximada** para modelos estadísticos
- **Eliminar tendencia** (diferenciación)

Exploramos cada una y evaluamos su efecto sobre la distribución.

In [14]:
# Transformación Log (log1p para evitar log(0))
df_clean["Caudal_log"] = np.log1p(df_clean["Caudal"])

# Transformación Box-Cox (requiere valores > 0)
caudal_positivo = df_clean["Caudal"].clip(lower=0.001)
df_clean["Caudal_boxcox"], lambda_bc = sp_stats.boxcox(caudal_positivo)
print(f"📊 Parámetro λ de Box-Cox: {lambda_bc:.4f}")

# Diferenciación de primer orden (eliminar tendencia)
df_clean["Caudal_diff"] = df_clean["Caudal"].diff()

# Comparar distribuciones con histogramas
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=["Original", "Log(1+x)", f"Box-Cox (λ={lambda_bc:.2f})", "Diferenciación (Δ)"])

fig.add_trace(go.Histogram(x=df_clean["Caudal"], nbinsx=50,
              marker_color="#1f77b4", name="Original"), row=1, col=1)
fig.add_trace(go.Histogram(x=df_clean["Caudal_log"], nbinsx=50,
              marker_color="#2ca02c", name="Log"), row=1, col=2)
fig.add_trace(go.Histogram(x=df_clean["Caudal_boxcox"], nbinsx=50,
              marker_color="#d62728", name="Box-Cox"), row=2, col=1)
fig.add_trace(go.Histogram(x=df_clean["Caudal_diff"].dropna(), nbinsx=50,
              marker_color="#9467bd", name="Diff"), row=2, col=2)

fig.update_layout(title="Distribución del Caudal bajo Diferentes Transformaciones",
                  width=1000, height=600, showlegend=False)
fig.show()

# Asimetría de cada transformación
print("\n📐 Asimetría (skewness) por transformación:")
print(f"  Original:       {df_clean['Caudal'].skew():.3f}")
print(f"  Log(1+x):       {df_clean['Caudal_log'].skew():.3f}")
print(f"  Box-Cox:         {pd.Series(df_clean['Caudal_boxcox']).skew():.3f}")
print(f"  Diferenciación:  {df_clean['Caudal_diff'].skew():.3f}")

📊 Parámetro λ de Box-Cox: 0.6583



📐 Asimetría (skewness) por transformación:
  Original:       0.983
  Log(1+x):       -0.820
  Box-Cox:         0.122
  Diferenciación:  1.870


In [15]:
# Serie transformada Log vs Original (temporal)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Caudal Original (m³/s)", "Caudal Transformado log(1+x)"],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["Caudal"],
              mode="lines", name="Original", line=dict(color="#1f77b4", width=0.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["Caudal_log"],
              mode="lines", name="Log(1+x)", line=dict(color="#2ca02c", width=0.8)), row=2, col=1)

fig.update_layout(title="Efecto de la Transformación Logarítmica sobre la Serie",
                  width=1000, height=500, showlegend=True)
fig.update_yaxes(title_text="m³/s", row=1, col=1)
fig.update_yaxes(title_text="log(1+Caudal)", row=2, col=1)
fig.show()

## 10. Verificación Final de la Serie Limpia

Revisamos que la serie resultante:
- No tenga valores `NaN`
- Tenga frecuencia diaria continua
- Estadísticas coherentes con la serie original

In [16]:
# Verificación final
print("=" * 60)
print("✅ VERIFICACIÓN DE LA SERIE LIMPIA")
print("=" * 60)
print(f"   Período:        {df_clean.index.min().date()} → {df_clean.index.max().date()}")
print(f"   Total registros: {len(df_clean)}")
print(f"   Frecuencia:     {df_clean.index.freq}")
print(f"   NaN residuales: {df_clean['Caudal'].isna().sum()}")
print(f"   Tipo de índice: {type(df_clean.index).__name__}")

# Comparar estadísticas: original vs limpia
orig_stats = df["Caudal"].describe()
clean_stats = df_clean["Caudal"].describe()

comparacion = pd.DataFrame({
    "Original (con gaps)": orig_stats,
    "Limpia (imputada+capped)": clean_stats,
    "Diferencia %": ((clean_stats - orig_stats) / orig_stats * 100).round(2),
})
print("\n📊 Comparación estadística:")
display(comparacion)

✅ VERIFICACIÓN DE LA SERIE LIMPIA
   Período:        2010-01-01 → 2017-12-31
   Total registros: 2922
   Frecuencia:     <Day>
   NaN residuales: 0
   Tipo de índice: DatetimeIndex

📊 Comparación estadística:


,Original (con gaps),Limpia (imputada+capped),Diferencia %
count,2653.000000,2922.000000,10.14
mean,3.402709,3.466200,1.87
std,1.614810,1.662589,2.96
min,0.081771,0.149659,83.02
25%,2.652833,2.633083,-0.74
50%,3.330000,3.336000,0.18
75%,3.983583,4.058508,1.88
max,16.150000,9.388000,-41.87


In [17]:
# Serie final limpia
fig = px.line(
    df_clean.reset_index(),
    x="Fecha",
    y="Caudal",
    title="Serie de Caudal Limpia: Imputada + Capped (2010–2017)",
    labels={"Caudal": "Caudal (m³/s)", "Fecha": ""},
)
fig.update_traces(line=dict(width=0.8, color="#2ca02c"))
fig.update_layout(
    width=1000, height=500,
    xaxis=dict(rangeslider=dict(visible=True)),
    hovermode="x unified",
)
fig.show()

## 11. Exportar Dataset Limpio para la Semana 3

Guardamos la serie limpia en CSV para uso directo en el EDA avanzado y modelado de las siguientes semanas.

In [18]:
# Preparar DataFrame para exportar (solo columnas útiles)
df_export = df_clean[["Caudal", "Caudal_log"]].copy()

# Guardar CSV limpio
output_path = "caudal_limpio_diario.csv"
df_export.to_csv(output_path)

print(f"✅ Dataset exportado: {output_path}")
print(f"   Filas: {len(df_export)}")
print(f"   Columnas: {list(df_export.columns)}")
print(f"   Período: {df_export.index.min().date()} → {df_export.index.max().date()}")
print(f"   Frecuencia: diaria (D)")
df_export.head()

✅ Dataset exportado: caudal_limpio_diario.csv
   Filas: 2922
   Columnas: ['Caudal', 'Caudal_log']
   Período: 2010-01-01 → 2017-12-31
   Frecuencia: diaria (D)


,Caudal,Caudal_log
Fecha,,
2010-01-01,1.196,0.786638
2010-01-02,1.196,0.786638
2010-01-03,1.196,0.786638
2010-01-04,1.196,0.786638
2010-01-05,1.196,0.786638


---

## Conclusiones de la Semana 2

### Hallazgos Clave:

1. **Reindexación:** Al forzar frecuencia diaria, se revelaron ~269 días faltantes (completitud ~90.8%).
2. **Patrón de gaps:** Los datos faltantes no son aleatorios — se concentran en ciertos meses/años, lo que sugiere fallas en la estación de medición.
3. **Imputación:** Se compararon 4 métodos (ffill, interpolación lineal, temporal, media móvil). Se seleccionó **interpolación lineal** por preservar mejor las estadísticas originales y generar transiciones suaves.
4. **Outliers:** Se detectaron valores extremos altos mediante IQR y Z-score. Se aplicó **capping (P1-P99)** en lugar de eliminación, ya que los picos de caudal son eventos hidrológicos legítimos.
5. **Transformaciones:** La transformación **log(1+x)** reduce significativamente la asimetría positiva y estabiliza la varianza — será útil para modelos estadísticos en semanas posteriores.
6. **Dataset exportado:** Serie diaria completa, imputada y con capping, lista para EDA avanzado.

### Próximos pasos (Semana 3):
- Descomposición estacional (trend, seasonal, residual)
- Análisis de autocorrelación (ACF/PACF)
- Pruebas de estacionariedad (ADF, KPSS)
- Formulación de pregunta de investigación
- Análisis de estacionalidad y patrones interanuales

In [ ]:
print(f"p01={p01:.4f} p99={p99:.4f} capped={capped}")
print(f"lambda_bc={lambda_bc:.4f}")
for y in pivot_comp.index:
    vals = [round(v,1) if not pd.isna(v) else 0 for v in pivot_comp.loc[y].tolist()]
    print(f"COMP_{y}: {vals}")

In [ ]:
for y in pivot_comp.index:
    vals = [round(v,1) for v in pivot_comp.loc[y].tolist()]
    print(f"C{y}={vals}")